# 04. Dataset Improvements

This notebook applies the cleanup decisions that came from the first full dataset build and the EDA.

The goal here is not to rebuild the dataset from scratch. The goal is to make the finished catalog cleaner before recommendation, ranking, graph analysis, and later modeling steps use it.

Main cleanup decisions:

- remove entries with no usable air date
- fill missing or zero episode counts from the AniDB cache, with optional live repair cells
- remove MAL `Special` / `TV Special` rows with no AniDB id
- remove duplicate specials or recaps that share an AniDB entry with a stronger canonical title
- add removed low-value `Special` / `TV Special` MAL ids to `data/build/skipped_invalid_type_ids.json` so the next rebuild can skip them early
- redirect relation and recommendation edges away from removed duplicate specials
- promote MAL-genre-equivalent tags into `genres` only when the AniDB tag weight is at least 400
- keep MAL themes as protected descriptive `tags`
- prune noisy, vague, low-frequency, and zero-weight AniDB normal tags
- keep explicit tags exempt from the low-frequency and weight-zero pruning rules
- merge AniDB `similaranime` links into recommendations only when approval is at least 25%

Live AniDB calls are separated into optional cells so a normal run stays cache-first and does not trigger unexpected requests.

## Setup

The command-line equivalent is:

```bash
python src/04_apply_dataset_improvements.py
```

The notebook uses the same helper functions as the script, but shows the decision logic and previews before saving the improved `anime_dataset.csv` and `anime_dataset.json`.

In [51]:
from pathlib import Path
import sys
import importlib.util

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

MODULE_PATH = ROOT / "src" / "04_apply_dataset_improvements.py"
spec = importlib.util.spec_from_file_location("dataset_improvements", MODULE_PATH)
improve = importlib.util.module_from_spec(spec)
spec.loader.exec_module(improve)

DATASET_CSV = ROOT / "data" / "processed" / "anime_dataset.csv"
DATASET_JSON = ROOT / "data" / "processed" / "anime_dataset.json"
ANIDB_CACHE = ROOT / "data" / "caches" / "anidb_metadata_cache.json"

df = pd.read_csv(DATASET_CSV)
cache_payload = improve.load_anidb_cache()

print(f"Dataset rows: {len(df):,}")
print(f"AniDB cache entries: {len(cache_payload.get('items', {})):,}")


Dataset rows: 14,991
AniDB cache entries: 15,728


## Baseline Null Audit

This audit treats real nulls, empty strings, and string placeholders such as `null` as missing. That matters for this dataset because values came from multiple APIs and file formats.


In [52]:
audit_columns = [
    "aired_year",
    "aired_month",
    "episodes",
    "season",
    "duration",
    "total_watch_minutes",
    "genres",
    "studios",
    "tags",
    "explicit_tags",
    "demographics",
]

missing_rows = []
for column in audit_columns:
    if column in {"aired_year", "aired_month", "episodes", "duration", "total_watch_minutes"}:
        missing = df[column].isna()
        if column == "episodes":
            missing = missing | (pd.to_numeric(df[column], errors="coerce").fillna(-1) == 0)
    elif column == "explicit_tags":
        missing = df[column].apply(improve.is_missing_text) & df.apply(improve.should_have_explicit_tags, axis=1)
    else:
        missing = df[column].apply(improve.is_missing_text)
    missing_rows.append({"column": column, "missing_or_zero": int(missing.sum())})

pd.DataFrame(missing_rows)


,column,missing_or_zero
0,aired_year,14
1,aired_month,14
2,episodes,8
3,season,14
4,duration,12
5,total_watch_minutes,20
6,genres,535
7,studios,1998
8,tags,2770
9,explicit_tags,206


## Rule 1: Remove Entries With No Air Date

Rows with both `aired_year` and `aired_month` missing are removed. There are very few of them, and without a date they are weak for temporal features, season inference, and milestone reporting.


In [53]:
missing_air_date = df["aired_year"].isna() & df["aired_month"].isna()
df.loc[
    missing_air_date,
    ["mal_id", "title", "type", "score", "members", "aired_year", "aired_month"],
].sort_values("mal_id")


,mal_id,title,type,score,members,aired_year,aired_month
7187,30059,Saru Kani Gassen,Movie,5.01,620,NaN,NaN
7222,30158,Okore!! Nonkuro,Special,5.55,338,NaN,NaN
7223,30159,Super Taromu,Special,5.14,391,NaN,NaN
7729,32237,Burutabu-chan,TV,5.39,383,NaN,NaN
7974,33187,Katsudou Shashin,Movie,5.50,9176,NaN,NaN
8553,35129,Tottoko Hamtarou no Tottoko Taisetsu!! Koutsuu...,OVA,6.35,761,NaN,NaN
8668,35494,Haha wo Tazunete Sanzenri (Special),Special,6.01,481,NaN,NaN
8697,35628,Honoo no Alpenrose: Ai no Symphony Ongaku-hen,OVA,5.92,628,NaN,NaN
9738,39102,Transformers: Choujin Master Force Soushuuhen,OVA,5.73,558,NaN,NaN
9744,39162,Tatakae! Chou Robot Seimeitai Transformers Vic...,OVA,5.69,515,NaN,NaN


## Rule 1b: Fill Season From Air Month

`season` is redundant with `aired_month` when the month exists, so empty season values are filled deterministically:

- January to March: `winter`
- April to June: `spring`
- July to September: `summer`
- October to December: `fall`


In [54]:
season_preview = df[["mal_id", "title", "aired_month", "season"]].copy()
season_preview["season_from_month"] = season_preview["aired_month"].apply(improve.infer_season)

season_preview[
    season_preview["season"].apply(improve.is_missing_text)
    & season_preview["season_from_month"].notna()
].head(25)


,mal_id,title,aired_month,season,season_from_month


## Rule 1c: Convert Duration to Minutes and Add Total Watch Time

The raw duration text is useful for inspection, but numeric modeling needs minutes. This step replaces `duration` with numeric minutes and adds:

`total_watch_minutes = episodes * duration`

Unknown durations stay null.


In [55]:
runtime_preview = df[["mal_id", "title", "episodes", "duration"]].copy()
runtime_preview["duration_minutes"] = runtime_preview["duration"].apply(improve.parse_duration_minutes)
runtime_preview["total_watch_minutes"] = (
    pd.to_numeric(runtime_preview["episodes"], errors="coerce")
    * pd.to_numeric(runtime_preview["duration_minutes"], errors="coerce")
)

runtime_preview.head(25)


,mal_id,title,episodes,duration,duration_minutes,total_watch_minutes
0,1,Cowboy Bebop,26.0,24.0,24.0,624.0
1,5,Cowboy Bebop: Tengoku no Tobira,1.0,115.0,115.0,115.0
2,6,Trigun,26.0,24.0,24.0,624.0
3,7,Witch Hunter Robin,26.0,25.0,25.0,650.0
4,8,Bouken Ou Beet,52.0,23.0,23.0,1196.0
5,15,Eyeshield 21,145.0,23.0,23.0,3335.0
6,16,Hachimitsu to Clover,24.0,23.0,23.0,552.0
7,17,Hungry Heart: Wild Striker,52.0,23.0,23.0,1196.0
8,18,Initial D Fourth Stage,24.0,27.0,27.0,648.0
9,19,Monster,74.0,24.0,24.0,1776.0


## Rule 2: Fill Missing Episode Counts From AniDB

MAL sometimes has `episodes = null` or `episodes = 0` for currently airing or recently announced titles. AniDB can fill some of these from the local cache. Anything still missing after the cache pass becomes a live-call candidate.


In [56]:
episode_gap = df["episodes"].isna() | (pd.to_numeric(df["episodes"], errors="coerce").fillna(-1) == 0)
episode_preview = df.loc[
    episode_gap,
    ["mal_id", "anidb_id", "title", "type", "status", "episodes", "popularity"],
].copy()

items = cache_payload.get("items", {})
episode_preview["cached_episode_count"] = episode_preview["anidb_id"].apply(
    lambda value: (
        items.get(str(int(value)), {}).get("episode_count")
        if pd.notna(value)
        else None
    )
)
episode_preview.sort_values(["cached_episode_count", "popularity"], ascending=[False, True])


,mal_id,anidb_id,title,type,status,episodes,popularity,cached_episode_count
13112,60988,NaN,Tian Guan Cifu Short Films,ONA,Currently Airing,NaN,6806,None
13379,63830,NaN,Re:Zero kara Hajimeru Break Time 4th Season,Special,Currently Airing,NaN,8190,None
11018,48956,NaN,Wu Shang Shen Di 2nd Season,ONA,Currently Airing,NaN,10705,None
8716,35686,NaN,Doraemon (2005) Specials,TV Special,Currently Airing,NaN,12201,None
11628,51768,NaN,Girigiri Warukunai Watame,ONA,Currently Airing,NaN,13157,None
11656,51870,NaN,Pokemon de Manabi Asobu,ONA,Currently Airing,NaN,14031,None
11653,51866,NaN,CoroCoro Monster Ball,ONA,Currently Airing,NaN,15657,None
7729,32237,NaN,Burutabu-chan,TV,Finished Airing,NaN,20201,None


## Rule 2b: Remove Low-Value Specials and Recaps

MAL and AniDB catalog anime differently. MAL often creates separate entries for small specials, recap episodes, and TV specials, while AniDB often groups them inside the main anime entry.

This rule removes two weak cases:

- `Special` / `TV Special` rows with no AniDB id
- duplicate specials or recap-like rows that share an AniDB id with a stronger canonical title

The canonical title is selected by preferring stronger catalog entries: non-special when available, then higher members, more episodes, and better popularity rank. This keeps important entries such as `Nekomonogatari: Kuro` while removing `Nekomonogatari: Kuro Recap`.

The removed MAL ids are also written into `data/build/skipped_invalid_type_ids.json` using the compact `mal_id`, `anime_type`, `index` format. They are still valid MAL types globally, but invalid for this project catalog because they are low-value duplicates or orphan specials.

In [57]:
no_anidb_clean_preview, no_anidb_specials_preview = improve.remove_specials_without_anidb(df)

duplicate_clean_preview, removed_to_canonical_preview, duplicate_specials_preview = improve.remove_duplicate_specials_with_shared_anidb(
    no_anidb_clean_preview
)

print(f"Special/TV Special rows without AniDB id: {len(no_anidb_specials_preview):,}")
print(f"Duplicate special/recap rows sharing AniDB ids: {len(duplicate_specials_preview):,}")
print(f"Rows after these removals: {len(duplicate_clean_preview):,}")

if not no_anidb_specials_preview.empty:
    display(
        no_anidb_specials_preview[
            ["mal_id", "title", "type", "members", "popularity"]
        ].sort_values("members", ascending=False).head(25)
    )

if not duplicate_specials_preview.empty:
    display(
        duplicate_specials_preview.sort_values(
            ["shared_anidb_id", "removed_members"],
            ascending=[True, False],
        ).head(50)
    )

Special/TV Special rows without AniDB id: 7
Duplicate special/recap rows sharing AniDB ids: 6
Rows after these removals: 14,978


,mal_id,title,type,members,popularity
13379,63830,Re:Zero kara Hajimeru Break Time 4th Season,Special,7456,8190
8716,35686,Doraemon (2005) Specials,TV Special,2310,12201
13387,64359,Meitantei Conan: Hanamaru na Answer,TV Special,1739,13150
13378,63806,Fate/Grand Order: Fujimaru Ritsuka wa Wakarana...,Special,947,15528
7223,30159,Super Taromu,Special,391,20076
7222,30158,Okore!! Nonkuro,Special,338,20864
10138,40641,Little Wonders: Sneeze,Special,284,21855


,removed_mal_id,removed_title,removed_type,removed_members,removed_episodes,shared_anidb_id,canonical_mal_id,canonical_title,canonical_type,canonical_members,canonical_episodes
0,42245,Tiger Mask Pilot Film,Special,345,1,864,3009,Tiger Mask,TV,11979,105
1,39162,Tatakae! Chou Robot Seimeitai Transformers Vic...,OVA,515,6,2065,926,Tatakae! Chou Robot Seimeitai Transformers Vic...,TV,3891,38
2,39102,Transformers: Choujin Master Force Soushuuhen,OVA,558,4,2069,924,Transformers: Choujin Master Force,TV,4139,43
3,40244,Fushigi no Kuni no Alice Specials,Special,435,2,2744,2572,Fushigi no Kuni no Alice,TV,4141,24
4,34960,Tesapuru da yo! Schedule no Au Hito dake Shuugou!,Special,678,6,10974,28835,Tesagure! Bukatsumono Spin-off Purupurun Sharu...,TV,4883,12
5,63503,Isekai no Sata wa Shachiku Shidai: Omoi wo Has...,Special,5506,1,19507,60226,Isekai no Sata wa Shachiku Shidai,TV,53026,12


## Rule 2c: Redirect or Drop Edges Pointing to Removed Specials

Removing duplicate specials is not enough by itself. Recommendation and relation fields may still point to the removed MAL ids.

For duplicate-special removals, edges are redirected to the canonical MAL id. If the redirect would create a self-link, it is dropped. For specials removed because they have no AniDB id, there is no safe canonical target, so edges pointing to them are dropped.

In [58]:
edge_rewrite_preview, edge_rows_changed_preview = improve.rewrite_edges_after_removals(
    duplicate_clean_preview.copy(),
    removed_to_canonical_preview,
)

print(f"Rows with relation/recommendation edge rewrites: {edge_rows_changed_preview:,}")

edge_compare = duplicate_clean_preview[["mal_id", "title", "relations", "recommendations"]].copy()
edge_compare = edge_compare.merge(
    edge_rewrite_preview[["mal_id", "relations", "recommendations"]],
    on="mal_id",
    suffixes=("_before", "_after"),
)
edge_compare = edge_compare[
    (edge_compare["relations_before"].fillna("").astype(str) != edge_compare["relations_after"].fillna("").astype(str))
    | (edge_compare["recommendations_before"].fillna("").astype(str) != edge_compare["recommendations_after"].fillna("").astype(str))
]

edge_compare.head(30)

Rows with relation/recommendation edge rewrites: 2,665


,mal_id,title,relations_before,recommendations_before,relations_after,recommendations_after
0,1,Cowboy Bebop,Side Story:5|Side Story:17205|Summary:4037,205:121|6:94|889:55|400:47|20057:40|2251:29|40...,Side Story:5,205:121|6:94|889:55|400:47|20057:40|2251:29|40...
5,15,Eyeshield 21,Side Story:1317|Side Story:6418,1604:15|263:10|11771:8|22:8|5040:7|1559:5|2112...,,1604:15|263:10|11771:8|22:8|5040:7|1559:5|2112...
6,16,Hachimitsu to Clover,Sequel:1142|Side Story:644,31646:18|1698:17|13759:13|6045:7|4081:6|877:6|...,Sequel:1142,31646:18|1698:17|13759:13|6045:7|4081:6|877:6|...
9,19,Monster,Summary:1109|Summary:39332,1535:117|13601:18|239:13|23283:13|35737:11|820...,,1535:117|13601:18|239:13|23283:13|35737:11|820...
10,20,Naruto,Sequel:1735|Side Story:761|Side Story:594|Side...,34572:58|21:47|11061:46|269:33|40748:32|6702:3...,Sequel:1735|Side Story:442|Side Story:936|Side...,34572:58|21:47|11061:46|269:33|40748:32|6702:3...
11,21,One Piece,Side Story:466|Side Story:459|Side Story:1094|...,6702:76|11061:65|20:47|918:24|223:19|10033:17|...,Side Story:466|Side Story:459|Side Story:460|S...,6702:76|11061:65|20:47|918:24|223:19|10033:17|...
12,22,Tennis no Oujisama,Sequel:995|Side Story:1190|Side Story:5996|Sid...,11771:16|21185:9|15:8|627:6|20583:4|28171:4|26...,Sequel:995|Side Story:1190|Side Story:815|Side...,11771:16|21185:9|15:8|627:6|20583:4|28171:4|26...
16,26,Texhnolyze,Summary:41068,790:26|339:19|2216:8|267:7|4981:7|885:4|202:4|...,,790:26|339:19|2216:8|267:7|4981:7|885:4|202:4|...
20,30,Shinseiki Evangelion,Sequel:32|Spin-Off:4130|Summary:31|Alternative...,9756:62|339:57|35849:42|165:40|2001:35|1690:31...,Sequel:32|Spin-Off:4130|Summary:31|Alternative...,9756:62|339:57|35849:42|165:40|2001:35|1690:31...
22,32,"Shinseiki Evangelion Movie: Air/Magokoro wo, K...",Prequel:30|Summary:31,11981:18|35120:11|47:9|2761:6|441:6|7311:4|437...,Prequel:30|Summary:31,11981:18|35120:11|47:9|2761:6|441:6|7311:4|437...


## Optional Live AniDB Repair

Run these cells only when you are ready to make live AniDB HTTP calls. They update the local AniDB cache first, then the normal improvement step uses the refreshed cache.

The order matters because live AniDB requests are valuable:

1. repair missing/zero episode counts first
2. then repair tags, explicit tags, and demographics by MAL popularity

Each successful AniDB response is saved immediately to `data/caches/anidb_metadata_cache.json`.


In [59]:
# Step 1: episode repair candidates only.
# This catches cached AniDB episode counts that are missing OR still zero.

episode_live_candidates = improve.episode_live_candidate_frame(df, cache_payload)
episode_candidate_ids = episode_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Live episode candidates: {len(episode_candidate_ids)}")
display(episode_live_candidates)

# Uncomment to spend live AniDB calls on episode gaps.
updated = improve.update_cache_with_live_payloads(
     cache_payload,
     episode_candidate_ids,
     label="episode_repair",
 )
print(f"Live AniDB episode cache updates: {updated}")


Live episode candidates: 0


,mal_id,anidb_id,title,type,status,episodes,popularity,cached_episode_count


Live AniDB episode cache updates: 0


### Optional Live AniDB Duration Repair

Run this after episode repair. It targets rows where `duration` or `total_watch_minutes` is still missing and sorts by MAL popularity.


In [60]:
# Step 2: duration / total watch-time repair candidates.

duration_live_candidates = improve.duration_live_candidate_frame(df)
duration_candidate_ids = duration_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Live duration/runtime candidates: {len(duration_candidate_ids)}")
display(duration_live_candidates.head(100))

# Uncomment to spend live AniDB calls on duration/runtime gaps.
updated = improve.update_cache_with_live_payloads(
     cache_payload,
     duration_candidate_ids,
     label="duration_runtime_repair",
 )
print(f"Live AniDB duration/runtime cache updates: {updated}")


Live duration/runtime candidates: 7


,mal_id,anidb_id,title,type,status,episodes,duration,total_watch_minutes,popularity,needs_duration,needs_total_watch_minutes
4851,10742,5384.0,Saru to Kani no Gassen,Movie,Finished Airing,1.0,NaN,NaN,13984,True,True
4857,10758,5386.0,Momotarou,Movie,Finished Airing,1.0,NaN,NaN,15761,True,True
6364,22511,6922.0,Kobutori,Movie,Finished Airing,1.0,NaN,NaN,16176,True,True
4855,10756,6914.0,Shita-kiri Suzume,Movie,Finished Airing,1.0,NaN,NaN,16365,True,True
4858,10759,6920.0,Kintarou,Movie,Finished Airing,1.0,NaN,NaN,16452,True,True
6463,23191,6917.0,Chokin no Susume,Movie,Finished Airing,1.0,NaN,NaN,16747,True,True
6633,24573,6912.0,Neko to Nezumi,Movie,Finished Airing,1.0,NaN,NaN,16862,True,True


[1/7] duration_runtime_repair | AniDB 5384 | request_start


[1/7] duration_runtime_repair | AniDB 5384 | no_update
[2/7] duration_runtime_repair | AniDB 5386 | request_start
[2/7] duration_runtime_repair | AniDB 5386 | no_update
[3/7] duration_runtime_repair | AniDB 6922 | request_start
[3/7] duration_runtime_repair | AniDB 6922 | no_update
[4/7] duration_runtime_repair | AniDB 6914 | request_start
[4/7] duration_runtime_repair | AniDB 6914 | no_update
[5/7] duration_runtime_repair | AniDB 6920 | request_start
[5/7] duration_runtime_repair | AniDB 6920 | no_update
[6/7] duration_runtime_repair | AniDB 6917 | request_start
[6/7] duration_runtime_repair | AniDB 6917 | no_update
[7/7] duration_runtime_repair | AniDB 6912 | request_start
[7/7] duration_runtime_repair | AniDB 6912 | no_update
Live AniDB duration/runtime cache updates: 0


### Optional Live AniDB Currently-Airing Refresh

Run this when you want fresh episode counts and metadata for currently-airing titles. It is sorted by popularity because these calls are expensive.


In [61]:
# Step 3: currently-airing refresh candidates.

airing_live_candidates = improve.currently_airing_update_candidate_frame(df)
airing_candidate_ids = airing_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Currently-airing live refresh candidates: {len(airing_candidate_ids)}")
display(airing_live_candidates.head(100))

# Uncomment to refresh currently-airing AniDB payloads.
# Use a limit if you only want the most popular currently-airing titles.
#updated = improve.update_cache_with_live_payloads(
#     cache_payload,
#     airing_candidate_ids,
#     limit=100,
#     label="currently_airing_refresh",
#)
#print(f"Live AniDB currently-airing cache updates: {updated}")


Currently-airing live refresh candidates: 205


,mal_id,anidb_id,title,type,status,episodes,duration,total_watch_minutes,popularity
11,21,69.0,One Piece,TV,Currently Airing,1163.0,24.0,27912.0,17
203,235,266.0,Meitantei Conan,TV,Currently Airing,1208.0,24.0,28992.0,728
11591,51553,17305.0,Tongari Boushi no Atelier,TV,Currently Airing,13.0,23.0,299.0,855
13162,61316,19242.0,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,TV,Currently Airing,19.0,23.0,437.0,1145
13180,61469,19287.0,Steel Ball Run: JoJo no Kimyou na Bouken,ONA,Currently Airing,1.0,47.0,47.0,1324
...,...,...,...,...,...,...,...,...,...
13337,62852,19674.0,Ghost Concert: Missing Songs,TV,Currently Airing,12.0,23.0,276.0,8596
12393,56524,18365.0,Tunshi Xingkong 4th Season,ONA,Currently Airing,175.0,21.0,3675.0,8775
11435,50855,17141.0,"Yamato yo, Towa ni: Rebel 3199",Movie,Currently Airing,22.0,110.0,2420.0,9225
13072,60651,19301.0,Yu-Gi-Oh! Card Game: The Chronicles,ONA,Currently Airing,14.0,6.0,84.0,9416


### Optional Live AniDB Metadata Repair

Run this only after episode repair. These candidates need tags, explicit tags, or demographics and are sorted by MAL popularity so the highest-impact titles are repaired first.


In [62]:
# Step 4: metadata repair candidates by scarcity, then popularity.

metadata_live_candidates = improve.metadata_live_candidate_frame(df)
metadata_candidate_ids = metadata_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Live tag/explicit/demographic/studio metadata candidates: {len(metadata_candidate_ids)}")
display(metadata_live_candidates.head(100).sort_values(["scarcity_priority", "popularity"], na_position="last"))

# Uncomment to spend live AniDB calls on metadata gaps.
# Use a small limit if you want to avoid burning too many AniDB requests in one sitting.
#updated = improve.update_cache_with_live_payloads(
#     cache_payload,
#     metadata_candidate_ids,
#     limit=100,
#     label="metadata_repair",
# )
#print(f"Live AniDB metadata cache updates: {updated}")


Live tag/explicit/demographic/studio metadata candidates: 5809


,mal_id,anidb_id,title,type,rating,popularity,tags,explicit_tags,demographics,duration,total_watch_minutes,studios,needs_tags,needs_explicit_tags,needs_demographics,needs_duration,needs_total_watch_minutes,needs_studios,scarcity_priority,need_count
4851,10742,5384.0,Saru to Kani no Gassen,Movie,G - All Ages,13984,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True,True,False,True,12,4
4857,10758,5386.0,Momotarou,Movie,G - All Ages,15761,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True,True,False,True,12,4
6364,22511,6922.0,Kobutori,Movie,G - All Ages,16176,Mythology,NaN,NaN,NaN,NaN,NaN,False,False,True,True,False,True,12,3
4855,10756,6914.0,Shita-kiri Suzume,Movie,G - All Ages,16365,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True,True,False,True,12,4
4858,10759,6920.0,Kintarou,Movie,G - All Ages,16452,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True,True,False,True,12,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4403,9318,6995.0,Samero,Movie,R+ - Mild Nudity,14523,NaN,NaN,NaN,3.0,3.0,Tsuji Naoyuki,True,True,True,False,False,False,206,3
6042,19921,5100.0,Ogami Matsugorou,OVA,R+ - Mild Nudity,14564,Delinquents|Martial Arts|School|delinquent|maf...,NaN,Shounen,46.0,46.0,NaN,False,True,False,False,False,True,206,2
5892,18569,2975.0,Taiman Blues: Shimizu Naoto-hen,OVA,R+ - Mild Nudity,14779,Delinquents|delinquent|friendship|School,NaN,Shounen,29.0,87.0,Shaft|Magic Bus,False,True,False,False,False,False,206,1
2214,2739,3980.0,Highschool Aurabuster: Hikari no Mezame,OVA,R+ - Mild Nudity,14792,Super Power,NaN,NaN,27.0,81.0,OLM,False,True,True,False,False,False,206,2


## Rule 3: Promote Genre-Equivalent Tags Into Genres

`genres` should contain broad MAL genre categories, while `tags` should describe more specific attributes.

This rule moves tag values that are equivalent to MAL genres into `genres` only when the AniDB tag weight is at least 400, then removes those high-confidence genre-equivalent tags from `tags`. Lower-weight genre-like tags stay as descriptive tags instead of becoming broad genres. Examples:

- `science fiction`, `hard science fiction`, `soft science fiction` -> `Sci-Fi`
- `daily life` -> `Slice of Life`
- `cooking` -> `Gourmet`
- `thriller` -> `Suspense`
- exact genre names such as `action`, `fantasy`, `romance`, and `comedy`

MAL themes stay protected as tags. Similar theme tags are canonicalized, for example `slapstick -> Gag Humor`, lowercase `gore -> Gore`, and `association football -> football`.

In [63]:
promoted_preview, promotion_summary = improve.normalize_tags_and_promote_genres(df)

changed_rows = df.index[
    (df["genres"].fillna("").astype(str) != promoted_preview["genres"].fillna("").astype(str))
    | (df["tags"].fillna("").astype(str) != promoted_preview["tags"].fillna("").astype(str))
]

print(promotion_summary)
promoted_preview.loc[
    changed_rows,
    ["mal_id", "title", "genres", "tags"],
].head(30)

{'genre_values_added_from_tags': 0, 'normal_tags_removed_as_genre_equivalents': 0, 'normal_tags_removed_as_noisy_or_vague': 0, 'normal_tags_canonicalized': 0, 'explicit_tags_canonicalized': 0}


,mal_id,title,genres,tags


## Rule 4: Fill Empty Studios With AniDB Origin Tags

If a studio is missing, AniDB `origin` tags can still describe production provenance. These are not real animation-studio names, so they are only used as a fallback when `studios` is empty.


In [64]:
studio_missing = df["studios"].apply(improve.is_missing_text)
studio_preview = df.loc[
    studio_missing,
    ["mal_id", "anidb_id", "title", "studios"],
].copy()
studio_preview["origin_fallback"] = studio_preview["anidb_id"].apply(
    lambda value: improve.origin_tags_from_payload(
        cache_payload.get("items", {}).get(str(int(value))) if pd.notna(value) else None
    )
)
studio_preview[
    studio_preview["origin_fallback"].apply(lambda value: not improve.is_missing_text(value))
].head(30)


,mal_id,anidb_id,title,studios,origin_fallback
502,548,1022.0,Wonderful Days,NaN,South Korean production
941,1099,2977.0,Shadow Skill: Kurudaryuu Kousatsuhou no Himitsu,NaN,Japanese production
1053,1223,2983.0,Ame to Shoujo to Watashi no Tegami,NaN,Japanese production
1146,1355,1591.0,Makyuu Senjou,NaN,Japanese production
1336,1585,3975.0,Spectral Force,NaN,Japanese production
1359,1611,2618.0,Galerians: Rion,NaN,Japanese production
1389,1656,2143.0,PostPet Momobin,NaN,Japanese production
1485,1776,1264.0,Bulg-eunmae,NaN,South Korean production
1520,1828,4926.0,Catblue: Dynamite,NaN,Japanese production
1576,1900,1156.0,Twin Signal: Family Game,NaN,Japanese production


## Rule 5: Refill Missing Tags and Demographics

For rows where `tags`, `explicit_tags`, or demographics are empty, the notebook reclassifies the raw AniDB tags from the cache using the same cleanup rules as the dataset builder and final tag cleanup:

- taxonomy containers are ignored
- demographics come from AniDB `target audience`
- explicit tags stay separate from recommender tags and are refilled for explicit-rated rows
- explicit maintenance labels such as `-- TO BE SPLIT` are removed before writing
- normal AniDB tags that are equivalent to MAL genres, such as `action` or `science fiction`, are not re-added to `tags`
- noisy/vague normal tags are not re-added
- tag weights are preserved for the cleaned tag names

If you edited `src/04_apply_dataset_improvements.py` while this notebook was already open, rerun the setup cell before rerunning this preview.

In [65]:
tag_or_demo_gap = (
    df["tags"].apply(improve.is_missing_text)
    | df["demographics"].apply(improve.is_missing_text)
    | (df["explicit_tags"].apply(improve.is_missing_text) & df.apply(improve.should_have_explicit_tags, axis=1))
)

tag_demo_preview = []
for _, row in df.loc[tag_or_demo_gap].head(40).iterrows():
    anidb_id = improve.parse_int(row.get("anidb_id"), default=None)
    payload = cache_payload.get("items", {}).get(str(anidb_id)) if anidb_id is not None else None
    parsed = improve.parse_anidb_tag_records((payload or {}).get("raw_tags", []), rating=row.get("rating"))
    tag_demo_preview.append(
        {
            "mal_id": row["mal_id"],
            "title": row["title"],
            "current_tags": row.get("tags"),
            "new_tags": parsed.get("tags"),
            "current_explicit_tags": row.get("explicit_tags"),
            "new_explicit_tags": parsed.get("explicit_tags"),
            "current_demographics": row.get("demographics"),
            "new_demographics": parsed.get("demographics"),
        }
    )

pd.DataFrame(tag_demo_preview)


,mal_id,title,current_tags,new_tags,current_explicit_tags,new_explicit_tags,current_demographics,new_demographics
0,55,Arc the Lad,bounty hunter|magic|multiple couples|angst|cri...,bounty hunter|magic|multiple couples|angst|cri...,NaN,,NaN,
1,69,Cluster Edge,Military|air force|high school|angel|gunfights...,Military|air force|high school|angel|gunfights...,NaN,,NaN,
2,75,Soukyuu no Fafner: Dead Aggressor,Mecha|Military|alien|robot|human enhancement|p...,Military|alien|robot|human enhancement|Mecha|p...,NaN,,NaN,
3,80,Kidou Senshi Gundam,Mecha|Military|Space|extrasensory perception|r...,Military|extrasensory perception|robot|swordpl...,NaN,,NaN,
4,83,Kidou Senshi Gundam: Dai 08 MS Shoutai - Mille...,Mecha|Military|robot|disaster|war|real robot,robot|Mecha|disaster|war|real robot,NaN,,NaN,
5,84,Kidou Senshi Gundam 0083: Stardust Memory,Mecha|Military|Space|robot|swordplay|space tra...,Military|robot|swordplay|Mecha|space travel|pi...,NaN,,NaN,
6,85,Kidou Senshi Zeta Gundam,Mecha|Military|Space|robot|swordplay|human enh...,Military|robot|swordplay|human enhancement|Mec...,NaN,,NaN,
7,86,Kidou Senshi Gundam ZZ,Mecha|Military|Space|child soldier|robot|colla...,Military|child soldier|robot|collateral damage...,NaN,,NaN,
8,87,Kidou Senshi Gundam: Gyakushuu no Char,Mecha|Military|Space|robot|human enhancement|s...,Military|robot|human enhancement|Mecha|space t...,NaN,,NaN,
9,88,Kidou Senshi Gundam F91,Mecha|Military|Space|robot|collateral damage|h...,Military|robot|collateral damage|human enhance...,NaN,,NaN,


## Rule 5b: Merge AniDB Similar-Anime Recommendations

MAL recommendations and AniDB `similaranime` both describe item-item similarity, but they use different ids and different voting systems.

This rule uses AniDB `similaranime` only when the approval ratio is at least 25 percent:

`approval / total >= 0.25`

AniDB ids are mapped back to MAL ids through the canonical AniDB-to-MAL map. When MAL and AniDB both suggest the same target, the larger weight is kept.

Extra guardrail: if multiple MAL rows share the same AniDB id, not every MAL row receives the shared AniDB recommendations. Minor rows such as recap, manner movie, omake, trailer, or very small side entries are skipped. `Summary` and `Full Story` are directional: a full entry may point to a recap with `Summary`, while a recap/compilation usually points back with `Full Story`. The rule keeps the full sequence entries eligible but avoids giving a one-minute side item the full recommendation neighborhood of the main work.

In [66]:
anidb_to_mal_preview = improve.canonical_mal_by_anidb(df)
shared_anidb_stats_preview = improve.anidb_group_stats(df)

similar_rows = []
skipped_shared_rows = []
for _, row in df[df["anidb_id"].notna()].iterrows():
    anidb_id = improve.parse_int(row.get("anidb_id"), default=None)
    payload = cache_payload.get("items", {}).get(str(anidb_id)) if anidb_id is not None else None
    new_edges = improve.similar_anime_edges_from_payload(payload, anidb_to_mal_preview)
    if not new_edges:
        continue

    eligible = improve.should_receive_anidb_similar_edges(row, shared_anidb_stats_preview)
    if not eligible:
        skipped_shared_rows.append(
            {
                "mal_id": row["mal_id"],
                "title": row["title"],
                "type": row.get("type"),
                "anidb_id": anidb_id,
                "members": row.get("members"),
                "duration": row.get("duration"),
                "relations": row.get("relations"),
                "approved_anidb_edges": "|".join(new_edges[:10]),
            }
        )
        continue

    merged = improve.merge_recommendation_edges(row.get("recommendations"), new_edges)
    before_count = len(improve.split_pipe_values(row.get("recommendations")))
    after_count = len(improve.split_pipe_values(merged))
    similar_rows.append(
        {
            "mal_id": row["mal_id"],
            "title": row["title"],
            "type": row.get("type"),
            "anidb_id": anidb_id,
            "approved_anidb_edges": "|".join(new_edges[:10]),
            "recommendation_count_before": before_count,
            "recommendation_count_after_merge": after_count,
            "would_add_edges": after_count - before_count,
        }
    )

similar_preview = pd.DataFrame(similar_rows).sort_values(
    ["would_add_edges", "recommendation_count_after_merge"],
    ascending=[False, False],
)
skipped_shared_preview = pd.DataFrame(skipped_shared_rows).sort_values(
    ["anidb_id", "members"],
    ascending=[True, False],
) if skipped_shared_rows else pd.DataFrame()

print(f"Rows eligible for approved AniDB similar-anime merge: {len(similar_preview):,}")
print(f"Rows skipped because shared AniDB id looks minor/non-representative: {len(skipped_shared_preview):,}")
display(similar_preview.head(40))
display(skipped_shared_preview.head(40))

Rows eligible for approved AniDB similar-anime merge: 3,358
Rows skipped because shared AniDB id looks minor/non-representative: 105


,mal_id,title,type,anidb_id,approved_anidb_edges,recommendation_count_before,recommendation_count_after_merge,would_add_edges
667,1531,Shakugan no Shana: Koi to Onsen no Kougai Gaku...,OVA,3408,1195:192|1691:91|4224:22|6682:20|356:13|4654:1...,0,11,11
2849,52608,Tensei Kizoku no Isekai Boukenroku: Jichou wo ...,TV,17552,36407:17|41312:13|49438:8|34497:7|38830:6|5261...,12,22,10
293,475,Hotori: Tada Saiwai wo Koinegau,TV Special,2842,7465:12|3701:7|36098:3|9213:3|7066:2|277:2|587...,9,19,10
253,395,Gantz 2nd Stage,TV,2400,6880:114|14345:72|34542:32|916:20|10620:12|160...,6,16,10
1502,9213,Kowarekake no Orgel,OVA,6271,59:55|27775:14|276:5|4483:4|23273:4|7465:4|277...,19,28,9
552,1141,Palme no Ki,Movie,604,34599:5|1140:4|47:3|1079:3|572:3|37515:2|2567:...,8,17,9
179,276,Mahoromatic: Automatic Maiden,TV,109,146:24|554:19|3085:18|27775:16|36516:11|375:8|...,23,31,8
622,1361,Final Fantasy: The Spirits Within,Movie,3146,6867:2|40613:2|731:1|1303:1|38749:1|4264:1|132...,6,14,8
585,1226,Seihou Tenshi Angel Links,TV,223,274:11|400:3|7066:2|276:2|277:2|587:1|23273:1|...,4,12,8
1163,4280,Kara no Kyoukai Movie 4: Garan no Dou,Movie,4932,169:225|5177:188|1556:30|3342:28|356:24|5356:1...,2,10,8


,mal_id,title,type,anidb_id,members,duration,relations,approved_anidb_edges
38,14685,Onegai☆Teacher: Reminiscence Disc,OVA,16,5553,57.0,NaN,11433:223|470:30|189:7|41389:7|6166:6|840:5
8,3247,Love Hina Final Selection,OVA,35,33708,28.0,NaN,53:72|591:45|157:37|193:28|539:17|519:11|18897...
76,39295,Hikaru no Go: Memories,OVA,84,1312,80.0,Alternative Version:41191,10800:64|31646:21|5671:12|2562:11|35180:7|427:...
36,13233,Mugen no Ryvius: Illusion,ONA,192,1204,7.0,NaN,342:41|113:29|39198:21|290:19|1901:16|2987:9|1...
24,8756,Bishoujo Senshi Sailor Moon Memorial,OVA,235,19257,90.0,NaN,1533:35|687:29|435:6
1,983,Cosplay Complex: Extra Identification,OVA,278,3453,5.0,Parent Story:982,615:1
4,2126,Yuu☆Yuu☆Hakusho: Eizou Hakusho II,OVA,312,14617,20.0,NaN,269:156|136:90|11061:75|813:61|238:44|38000:4
3,2125,Yuu☆Yuu☆Hakusho: Eizou Hakusho - Ankoku Bujuts...,OVA,312,14313,26.0,NaN,269:156|136:90|11061:75|813:61|238:44|38000:4
35,12455,Yuu☆Yuu☆Hakusho: Mu Mu Hakusho - Nightmare Hak...,OVA,312,11090,2.0,Parent Story:392,269:156|136:90|11061:75|813:61|238:44|38000:4
16,5165,Macross 25-shuunen Kinen: All That VF Macross ...,ONA,427,3974,2.0,Parent Story:3572,16524:39


## Rule 5c: Prune Noisy and Low-Signal Normal Tags

After tag refilling and genre promotion, normal tags are pruned for modeling quality.

Rules:

- MAL themes are protected and stay as tags
- explicit tags are exempt from frequency and weight-zero pruning
- normal AniDB tags with global count below 5 are removed
- normal AniDB tags with weight 0 are removed unless needed as a small fallback for tag-poor rows
- noisy or vague tags such as `cast`, `tropes`, `speculative fiction`, and `fire` are removed

The exported `tag_quality_review.csv` keeps the before/after counts and examples for manual inspection.

In [67]:
tag_prune_input, _tag_norm_summary = improve.normalize_tags_and_promote_genres(df)
tag_pruned_preview, tag_prune_summary = improve.prune_dataset_tags(tag_prune_input.copy())

print(tag_prune_summary)

changed_tag_rows = tag_prune_input.index[
    (tag_prune_input["tags"].fillna("").astype(str) != tag_pruned_preview["tags"].fillna("").astype(str))
]

tag_prune_compare = tag_prune_input.loc[
    changed_tag_rows,
    ["mal_id", "title", "tags", "tag_weights"],
].copy()
tag_prune_compare = tag_prune_compare.merge(
    tag_pruned_preview.loc[changed_tag_rows, ["mal_id", "tags", "tag_weights"]],
    on="mal_id",
    suffixes=("_before", "_after"),
)

tag_prune_compare.head(30)

{'unique_tags_before': 854, 'unique_tags_after': 355, 'tag_values_removed': 60806, 'rows_with_tags_changed': 7515}


,mal_id,title,tags_before,tag_weights_before,tags_after,tag_weights_after
0,1,Cowboy Bebop,Adult Cast|Space|bounty hunter|Detective|rever...,Adult Cast:0|Space:600|bounty hunter:600|Detec...,Adult Cast|Space|bounty hunter|Detective|rever...,Adult Cast:0|Space:600|bounty hunter:600|Detec...
1,5,Cowboy Bebop: Tengoku no Tobira,Adult Cast|Space|bounty hunter|gunfights|space...,Adult Cast:0|Space:600|bounty hunter:600|gunfi...,Adult Cast|Space|bounty hunter|gunfights|plot ...,Adult Cast:0|Space:600|bounty hunter:600|gunfi...
2,6,Trigun,Adult Cast|cyborg|alien|gunfights|human enhanc...,Adult Cast:0|cyborg:200|alien:400|gunfights:60...,Adult Cast|cyborg|alien|gunfights|Mecha|Gag Hu...,Adult Cast:0|cyborg:200|alien:400|gunfights:60...
3,7,Witch Hunter Robin,Detective|Military|painting|undead|immortality...,Detective:400|Military:100|painting:100|undead...,Detective|Military|painting|bishounen|magic|gh...,Detective:400|Military:100|painting:100|bishou...
4,8,Bouken Ou Beet,magic|Super Power|RPG aspects|time skip,magic:500|Super Power:400|RPG aspects:0|time s...,magic|Super Power,magic:500|Super Power:400
5,15,Eyeshield 21,Team Sports|American football|shota|high schoo...,Team Sports:600|American football:600|shota:10...,Team Sports|shota|high school|delinquent|Gag H...,Team Sports:600|shota:100|high school:400|deli...
6,16,Hachimitsu to Clover,Adult Cast|Love Polygon|Visual Arts|Gag Humor|...,Adult Cast:0|Love Polygon:400|Visual Arts:600|...,Adult Cast|Love Polygon|Visual Arts|Gag Humor|...,Adult Cast:0|Love Polygon:400|Visual Arts:600|...
7,17,Hungry Heart: Wild Striker,Team Sports|association football|high school|S...,Team Sports:600|association football:600|high ...,Team Sports|association football|high school|S...,Team Sports:600|association football:600|high ...
8,18,Initial D Fourth Stage,Racing|motorsport|street racing|plot continuit...,Racing:600|motorsport:600|street racing:400|pl...,Racing|motorsport|street racing|plot continuit...,Racing:600|motorsport:600|street racing:400|pl...
9,19,Monster,Adult Cast|Psychological|Detective|amnesia|gun...,Adult Cast:0|Psychological:0|Detective:500|amn...,Adult Cast|Psychological|Detective|gunfights|p...,Adult Cast:0|Psychological:0|Detective:500|gun...


## Rule 6: Conservative Demographic Inference

Demographics are the hardest remaining nulls because MAL often leaves them empty and AniDB only fills them when `target audience` tags exist.

This notebook fills only high-confidence cases:

- `Rx - Hentai`, `R+ - Mild Nudity`, `Hentai`, or `Erotica` -> `18+`
- `PG - Children` -> `Kodomo`
- exact demographic-like tags such as `shounen`, `seinen`, `shoujo`, `josei`, `kodomo`, or `mina`

It does not infer demographics from broad genres like action, comedy, romance, or fantasy because those are content categories, not audience categories.


In [68]:
demographic_gap = df["demographics"].apply(improve.is_missing_text)
demographic_preview = df.loc[
    demographic_gap,
    ["mal_id", "title", "rating", "genres", "tags", "explicit_tags", "demographics"],
].copy()
demographic_preview["inferred_demographics"] = demographic_preview.apply(
    improve.infer_demographics_from_row,
    axis=1,
)

demographic_preview[
    demographic_preview["inferred_demographics"].apply(lambda value: not improve.is_missing_text(value))
].head(40)


,mal_id,title,rating,genres,tags,explicit_tags,demographics,inferred_demographics
129,151,Re: Cutie Honey,R+ - Mild Nudity,Action|Comedy|Girls Love|Sci-Fi|Ecchi|Romance,android|cosplaying|cyborg|robot|Mahou Shoujo|c...,BDSM|nudity|violence|skimpy clothing|pantsu|po...,NaN,18+
173,197,Rizelmine,R+ - Mild Nudity,Comedy|Romance|Sci-Fi|Ecchi,School|sudden girlfriend appearance|Love Polyg...,loli|nudity|pantsu|ecchi,NaN,18+
243,275,Love♥Love?,R+ - Mild Nudity,Comedy|Romance|Ecchi,Harem|high school|Gag Humor|performance|School...,trap|small breasts|Crossdressing|loli|nudity|v...,NaN,18+
283,315,Xiao Qian,PG - Children,Action|Drama|Fantasy|Romance|Comedy,NaN,NaN,NaN,Kodomo
385,424,Dirty Pair,R+ - Mild Nudity,Adventure|Comedy|Sci-Fi|Gourmet|Action|Romance...,Military|bounty hunter|Detective|robot helper|...,lingerie|nudity|violence|skimpy clothing|pants...,NaN,18+
395,434,Legend of Lemnear: Kyokuguro no Tsubasa Valkisas,R+ - Mild Nudity,Action|Adventure|Fantasy,magic|dragon|swordplay|Super Power|demon|high ...,breast fondling|nudity|violence|skimpy clothin...,NaN,18+
449,492,Armitage III: Dual-Matrix,R+ - Mild Nudity,Sci-Fi|Action,Mecha|android|cyborg|robot|Martial Arts|gunfig...,violence|Gore,NaN,18+
450,493,Armitage III: Poly-Matrix,R+ - Mild Nudity,Mystery|Romance|Sci-Fi|Action|Suspense,Detective|Mecha|Military|android|robot|Martial...,violence|Gore,NaN,18+
459,504,Garou Densetsu: The Motion Picture,R+ - Mild Nudity,Action|Adventure|Drama|Romance,Martial Arts|fighting|damsel in distress,skimpy clothing|ecchi,NaN,18+
494,539,Tenchi Muyou! Ryououki,R+ - Mild Nudity,Action|Comedy|Sci-Fi|Fantasy|Romance,Space|high school|alien|space travel|demon|hum...,nudity|ecchi|Harem,NaN,18+


## Apply Improvements

This cell applies the same cleanup sequence previewed above and used by the command-line script. The previews make the decisions auditable; this final cell performs the save-ready transformation and reports what changed.

In [69]:
improved_df, summary = improve.apply_improvements(df, cache_payload)

summary_display = {
    key: value
    for key, value in summary.items()
    if not key.startswith("remaining_")
}
summary_display


{'started_rows': 14991,
 'dropped_missing_air_date': 14,
 'seasons_filled_from_aired_month': 0,
 'duration_values_parsed_to_minutes': 14958,
 'duration_filled_from_anidb_episode_lengths': 0,
 'total_watch_minutes_filled': 14953,
 'total_watch_minutes_filled_from_anidb_episode_lengths': 0,
 'episodes_filled_from_anidb_cache': 0,
 'studios_filled_from_origin_tags': 964,
 'recommendations_augmented_from_anidb_similar_anime': 1985,
 'anidb_similar_recommendation_rows_skipped_shared_minor': 228,
 'tags_filled_from_anidb_cache': 3,
 'explicit_tags_filled_from_anidb_cache': 14,
 'demographics_filled_from_anidb_cache': 5,
 'demographics_normalized': 0,
 'demographics_inferred_from_rating_genres_tags': 729,
 'duplicate_special_rows_removed': 2,
 'special_rows_without_anidb_removed': 5,
 'removed_special_ids_added_to_invalid_type_registry': 7,
 'edge_rows_rewritten_after_duplicate_special_removal': 2665,
 'duplicate_special_removal_audit_csv': 'C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\B

The following tables are the remaining cases that need live AniDB or manual review. They are kept as notebook output, not extra dataset files.


In [70]:
pd.DataFrame(summary["remaining_episode_live_candidates"]).head(50)


""


## Optional Manual Drop: Remaining Episode Gaps

After cache and optional live repair, any row still having `episodes` as null or zero can be removed if the remaining cases are negligible. The deletion lines are commented so a normal `Run All` does not silently drop rows.


In [71]:
episode_gap_after = (
    improved_df["episodes"].isna()
    | (pd.to_numeric(improved_df["episodes"], errors="coerce").fillna(-1) == 0)
)

episode_gap_rows = improved_df.loc[
    episode_gap_after,
    ["mal_id", "anidb_id", "title", "type", "status", "episodes", "popularity"],
].sort_values(["anidb_id", "popularity"], na_position="last")

print(f"Remaining episode gaps: {len(episode_gap_rows)}")
episode_gap_rows


Remaining episode gaps: 5


,mal_id,anidb_id,title,type,status,episodes,popularity
13112,60988,NaN,Tian Guan Cifu Short Films,ONA,Currently Airing,NaN,6806
11018,48956,NaN,Wu Shang Shen Di 2nd Season,ONA,Currently Airing,NaN,10705
11628,51768,NaN,Girigiri Warukunai Watame,ONA,Currently Airing,NaN,13157
11656,51870,NaN,Pokemon de Manabi Asobu,ONA,Currently Airing,NaN,14031
11653,51866,NaN,CoroCoro Monster Ball,ONA,Currently Airing,NaN,15657


In [72]:
# Manual deletion cell.
# Uncomment and run this cell only if you decide the remaining episode gaps are negligible.

episode_gap_after = (
     improved_df["episodes"].isna()
     | (pd.to_numeric(improved_df["episodes"], errors="coerce").fillna(-1) == 0)
 )
improved_df = improved_df.loc[~episode_gap_after].copy()
print(f"Rows after dropping episode gaps: {len(improved_df):,}")


Rows after dropping episode gaps: 14,965


In [73]:
pd.DataFrame(summary["remaining_tag_demographic_or_explicit_live_candidates"]).head(50)


,mal_id,anidb_id,title,tags,explicit_tags,demographics
0,55,433.0,Arc the Lad,bounty hunter|magic|multiple couples|angst|rev...,NaN,
1,69,2392.0,Cluster Edge,Military|air force|high school|angel|gunfights...,NaN,
2,75,1806.0,Soukyuu no Fafner: Dead Aggressor,Mecha|Military|alien|human enhancement|piloted...,NaN,
3,80,715.0,Kidou Senshi Gundam,Mecha|Military|Space|swordplay|Super Power|spa...,NaN,
4,83,1425.0,Kidou Senshi Gundam: Dai 08 MS Shoutai - Mille...,Mecha|Military|war|real robot,NaN,
5,84,717.0,Kidou Senshi Gundam 0083: Stardust Memory,Mecha|Military|Space|swordplay|space travel|pi...,NaN,
6,85,718.0,Kidou Senshi Zeta Gundam,Mecha|Military|Space|swordplay|human enhanceme...,NaN,
7,86,719.0,Kidou Senshi Gundam ZZ,Mecha|Military|Space|robot|gunfights|swordplay...,NaN,
8,87,720.0,Kidou Senshi Gundam: Gyakushuu no Char,Mecha|Military|Space|human enhancement|space t...,NaN,
9,88,626.0,Kidou Senshi Gundam F91,Mecha|Military|Space|human enhancement|space t...,NaN,


## Final Audit and Save

The final audit confirms the improved missingness profile. The save cell overwrites the processed dataset files because this notebook is the approved cleanup step after the full retry run.


In [74]:
final_missing_rows = []
for column in audit_columns:
    if column in {"aired_year", "aired_month", "episodes", "duration", "total_watch_minutes"}:
        missing = improved_df[column].isna()
        if column == "episodes":
            missing = missing | (pd.to_numeric(improved_df[column], errors="coerce").fillna(-1) == 0)
    elif column == "explicit_tags":
        missing = improved_df[column].apply(improve.is_missing_text) & improved_df.apply(improve.should_have_explicit_tags, axis=1)
    else:
        missing = improved_df[column].apply(improve.is_missing_text)
    final_missing_rows.append({"column": column, "missing_or_zero_after": int(missing.sum())})

pd.DataFrame(final_missing_rows)


,column,missing_or_zero_after
0,aired_year,0
1,aired_month,0
2,episodes,0
3,season,0
4,duration,12
5,total_watch_minutes,12
6,genres,531
7,studios,1021
8,tags,2787
9,explicit_tags,192


In [75]:
improve.save_dataset(improved_df, csv_path=DATASET_CSV, json_path=DATASET_JSON)

print(f"Saved {len(improved_df):,} rows to:")
print(DATASET_CSV)
print(DATASET_JSON)


Saved 14,965 rows to:
c:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\processed\anime_dataset.csv
c:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\processed\anime_dataset.json
